In [41]:
import torch
import torch.nn as nn

In [42]:
batch_size = 64
sequence_len = 64
embedding_dim = 32
layer_size = 256
steps = 5000

In [43]:
# input & output initialization
with open("inputs-shakespeare.txt", "r") as f:
    data = f.read()
chars = sorted(set(data))
vocab_size = len(chars)

ctoi = {c: i for i, c in enumerate(chars)}
itoc = {i: c for i, c in enumerate(chars)}

train = data[:int(len(data) * 0.9)]
val = data[int(len(data) * 0.9):]

x_train = torch.tensor([ctoi[ch] for ch in train[:-1]], dtype=torch.long).unfold(0, sequence_len, 1)
y_train = torch.tensor([ctoi[ch] for ch in train[1:]], dtype=torch.long).unfold(0, sequence_len, 1)

x_val = torch.tensor([ctoi[ch] for ch in val[:-1]], dtype=torch.long).unfold(0, sequence_len, sequence_len)
y_val = torch.tensor([ctoi[ch] for ch in val[1:]], dtype=torch.long).unfold(0, sequence_len, sequence_len)

In [44]:
# model declaration
class RecurrentNeuralNetwork(nn.Module):
    def __init__(self, vocab_size, embedding_dim, layer_size):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.input_linear = nn.Linear(in_features=embedding_dim, out_features=layer_size)
        self.hidden_linear = nn.Linear(in_features=layer_size, out_features=layer_size)
        self.output_linear = nn.Linear(in_features=layer_size, out_features=vocab_size)        

    def forward(self, input, h=None):
        batch_size = input.size(0)
        if h is None:
            h = torch.zeros((batch_size, layer_size))
        logits = []
        for i in range(input.size(1)):
            ch = input[:, i]
            new_h = torch.tanh(self.input_linear(self.embedding(ch)) + self.hidden_linear(h))
            logits.append(self.output_linear(new_h))
            h = new_h
        return torch.stack(logits, dim=1), h

In [45]:
# model initialization
model = RecurrentNeuralNetwork(vocab_size, embedding_dim, layer_size)
# optimizer initialization
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-3)

In [46]:
# training
for step in range(steps):
    optimizer.zero_grad()
    
    ix = torch.randint(0, len(x_train), (batch_size,))
    logits_train, _ = model(x_train[ix])
    loss_train = nn.functional.cross_entropy(logits_train.view(-1, vocab_size), y_train[ix].view(-1))

    if step % 1000 == 0 or step == steps - 1:
        with torch.no_grad():
            logits_val, _ = model(x_val)
            loss_val = nn.functional.cross_entropy(logits_val.view(-1, vocab_size), y_val.view(-1))
            print(f"Step {step}: train loss = {loss_train.item()}, val loss = {loss_val.item()}")

    loss_train.backward()
    optimizer.step()

Step 0: train loss = 4.236513137817383, val loss = 4.235915660858154
Step 1000: train loss = 1.5167828798294067, val loss = 1.722593903541565
Step 2000: train loss = 1.4471781253814697, val loss = 1.6743212938308716
Step 3000: train loss = 1.4705700874328613, val loss = 1.6576687097549438
Step 4000: train loss = 1.448925256729126, val loss = 1.6403042078018188
Step 4999: train loss = 1.3447353839874268, val loss = 1.6465885639190674


In [52]:
# prediction
with torch.inference_mode():
    output = ['R', 'O', 'M', 'E', 'O', ':', '\n']
    max_output_length = 1000
    h = None

    i = torch.tensor([[ctoi[ch] for ch in output]], dtype=torch.long)            

    while len(output) < max_output_length:
        logits, h = model(i, h)

        P = torch.softmax(logits[0, -1, :], dim=0)
        j = torch.multinomial(P, num_samples=1, replacement=True).item()
        output.append(itoc[j])

        i = torch.tensor([[ctoi[output[-1]]]], dtype=torch.long)

In [53]:
print(''.join(output))

ROMEO:
Not so, and captimoroust. King Bone 'what in Gloucester should not seen swill down this time on the sentry by woe not Pardon you by give me mall his these with thy consul, you know you here is no lord has break to effect than yet being alrious so much comm was a your wheir ever, when Did I have course,
Believe us well.

BRUTUS:
The first die deallancing and dead! Look, I, if I am given to each his triving thee froms
Of this butines,
This--

EDWARD:
'Twas behold him his businest-hates your house my things,
And pawh since the prazer, before
Can tell the gods,
And me their hand show?

QUEEN MARGARET:
So take
the honestless hot up town soges.

KING RICHARD III:
Mounty.

CORIOLANUS:
At the thing hast he within of bring the temptatorror:
The could like his, herea home on thee toe for him marratey-hambern'd of man do extriad me of the peofs thum rather the king is tedy?

DUKE
O, and but, and past death
A was all the part of Petize my soul,
Even to this side, and never liking.

Nursuani